# Subject: test_tools
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/cpython/Lib/test/test_tools

### File: __main__.py

In [ ]:
from test.test_tools import load_tests
import unittest

unittest.main()

### File: test_freeze.py

In [ ]:
"""Sanity-check tests for the "freeze" tool."""

import sys
import textwrap
import unittest

from test import support
from test.support import os_helper
from test.test_tools import imports_under_tool, skip_if_missing

skip_if_missing('freeze')
with imports_under_tool('freeze', 'test'):
    import freeze as helper

@support.requires_zlib()
@unittest.skipIf(sys.platform.startswith('win'), 'not supported on Windows')
@unittest.skipIf(sys.platform == 'darwin' and sys._framework,
        'not supported for frameworks builds on macOS')
@support.skip_if_buildbot('not all buildbots have enough space')
# gh-103053: Skip test if Python is built with Profile Guided Optimization
# (PGO), since the test is just too slow in this case.
@unittest.skipIf(support.check_cflags_pgo(),
                 'test is too slow with PGO')
class TestFreeze(unittest.TestCase):

    @support.requires_resource('cpu') # Building Python is slow
    def test_freeze_simple_script(self):
        script = textwrap.dedent("""
            import sys
            print('running...')
            sys.exit(0)
            """)
        with os_helper.temp_dir() as outdir:
            outdir, scriptfile, python = helper.prepare(script, outdir)
            executable = helper.freeze(python, scriptfile, outdir)
            text = helper.run(executable)
        self.assertEqual(text, 'running...')

### File: test_i18n.py

In [ ]:
"""Tests to cover the Tools/i18n package"""

import os
import re
import sys
import unittest
from textwrap import dedent
from pathlib import Path

from test.support.script_helper import assert_python_ok
from test.test_tools import imports_under_tool, skip_if_missing, toolsdir
from test.support.os_helper import temp_cwd, temp_dir


skip_if_missing()

DATA_DIR = Path(__file__).resolve().parent / 'i18n_data'


with imports_under_tool("i18n"):
    from pygettext import (parse_spec, process_keywords, DEFAULTKEYWORDS,
                           unparse_spec)


def normalize_POT_file(pot):
    """Normalize the POT creation timestamp, charset and
    file locations to make the POT file easier to compare.

    """
    # Normalize the creation date.
    date_pattern = re.compile(r'"POT-Creation-Date: .+?\\n"')
    header = r'"POT-Creation-Date: 2000-01-01 00:00+0000\\n"'
    pot = re.sub(date_pattern, header, pot)

    # Normalize charset to UTF-8 (currently there's no way to specify the output charset).
    charset_pattern = re.compile(r'"Content-Type: text/plain; charset=.+?\\n"')
    charset = r'"Content-Type: text/plain; charset=UTF-8\\n"'
    pot = re.sub(charset_pattern, charset, pot)

    # Normalize file location path separators in case this test is
    # running on Windows (which uses '\').
    fileloc_pattern = re.compile(r'#:.+')

    def replace(match):
        return match[0].replace(os.sep, "/")
    pot = re.sub(fileloc_pattern, replace, pot)
    return pot


class Test_pygettext(unittest.TestCase):
    """Tests for the pygettext.py tool"""

    script = Path(toolsdir, 'i18n', 'pygettext.py')

    def get_header(self, data):
        """ utility: return the header of a .po file as a dictionary """
        headers = {}
        for line in data.split('\n'):
            if not line or line.startswith(('#', 'msgid', 'msgstr')):
                continue
            line = line.strip('"')
            key, val = line.split(':', 1)
            headers[key] = val.strip()
        return headers

    def get_msgids(self, data):
        """ utility: return all msgids in .po file as a list of strings """
        msgids = []
        reading_msgid = False
        cur_msgid = []
        for line in data.split('\n'):
            if reading_msgid:
                if line.startswith('"'):
                    cur_msgid.append(line.strip('"'))
                else:
                    msgids.append('\n'.join(cur_msgid))
                    cur_msgid = []
                    reading_msgid = False
                    continue
            if line.startswith('msgid '):
                line = line[len('msgid '):]
                cur_msgid.append(line.strip('"'))
                reading_msgid = True
        else:
            if reading_msgid:
                msgids.append('\n'.join(cur_msgid))

        return msgids

    def assert_POT_equal(self, expected, actual):
        """Check if two POT files are equal"""
        self.maxDiff = None
        self.assertEqual(normalize_POT_file(expected), normalize_POT_file(actual))

    def extract_from_str(self, module_content, *, args=(), strict=True,
                         with_stderr=False, raw=False):
        """Return all msgids extracted from module_content."""
        filename = 'test.py'
        with temp_cwd(None):
            with open(filename, 'w', encoding='utf-8') as fp:
                fp.write(module_content)
            res = assert_python_ok('-Xutf8', self.script, *args, filename)
            if strict:
                self.assertEqual(res.err, b'')
            with open('messages.pot', encoding='utf-8') as fp:
                data = fp.read()
        if not raw:
            data = self.get_msgids(data)
        if not with_stderr:
            return data
        return data, res.err

    def extract_docstrings_from_str(self, module_content):
        """Return all docstrings extracted from module_content."""
        return self.extract_from_str(module_content, args=('--docstrings',), strict=False)

    def get_stderr(self, module_content):
        return self.extract_from_str(module_content, strict=False, with_stderr=True)[1]

    def test_header(self):
        """Make sure the required fields are in the header, according to:
           http://www.gnu.org/software/gettext/manual/gettext.html#Header-Entry
        """
        with temp_cwd(None) as cwd:
            assert_python_ok('-Xutf8', self.script)
            with open('messages.pot', encoding='utf-8') as fp:
                data = fp.read()
            header = self.get_header(data)

            self.assertIn("Project-Id-Version", header)
            self.assertIn("POT-Creation-Date", header)
            self.assertIn("PO-Revision-Date", header)
            self.assertIn("Last-Translator", header)
            self.assertIn("Language-Team", header)
            self.assertIn("MIME-Version", header)
            self.assertIn("Content-Type", header)
            self.assertIn("Content-Transfer-Encoding", header)
            self.assertIn("Generated-By", header)

            # not clear if these should be required in POT (template) files
            #self.assertIn("Report-Msgid-Bugs-To", header)
            #self.assertIn("Language", header)

            #"Plural-Forms" is optional

    @unittest.skipIf(sys.platform.startswith('aix'),
                     'bpo-29972: broken test on AIX')
    def test_POT_Creation_Date(self):
        """ Match the date format from xgettext for POT-Creation-Date """
        from datetime import datetime
        with temp_cwd(None) as cwd:
            assert_python_ok('-Xutf8', self.script)
            with open('messages.pot', encoding='utf-8') as fp:
                data = fp.read()
            header = self.get_header(data)
            creationDate = header['POT-Creation-Date']

            # peel off the escaped newline at the end of string
            if creationDate.endswith('\\n'):
                creationDate = creationDate[:-len('\\n')]

            # This will raise if the date format does not exactly match.
            datetime.strptime(creationDate, '%Y-%m-%d %H:%M%z')

    def test_output_option(self):
        for opt in ('-o', '--output='):
            with temp_cwd():
                assert_python_ok(self.script, f'{opt}test')
                self.assertTrue(os.path.exists('test'))
                res = assert_python_ok(self.script, f'{opt}-')
                self.assertIn(b'Project-Id-Version: PACKAGE VERSION', res.out)

    def test_funcdocstring(self):
        for doc in ('"""doc"""', "r'''doc'''", "R'doc'", 'u"doc"'):
            with self.subTest(doc):
                msgids = self.extract_docstrings_from_str(dedent('''\
                def foo(bar):
                    %s
                ''' % doc))
                self.assertIn('doc', msgids)

    def test_funcdocstring_bytes(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        def foo(bar):
            b"""doc"""
        '''))
        self.assertFalse([msgid for msgid in msgids if 'doc' in msgid])

    def test_funcdocstring_fstring(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        def foo(bar):
            f"""doc"""
        '''))
        self.assertFalse([msgid for msgid in msgids if 'doc' in msgid])

    def test_classdocstring(self):
        for doc in ('"""doc"""', "r'''doc'''", "R'doc'", 'u"doc"'):
            with self.subTest(doc):
                msgids = self.extract_docstrings_from_str(dedent('''\
                class C:
                    %s
                ''' % doc))
                self.assertIn('doc', msgids)

    def test_classdocstring_bytes(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        class C:
            b"""doc"""
        '''))
        self.assertFalse([msgid for msgid in msgids if 'doc' in msgid])

    def test_classdocstring_fstring(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        class C:
            f"""doc"""
        '''))
        self.assertFalse([msgid for msgid in msgids if 'doc' in msgid])

    def test_moduledocstring(self):
        for doc in ('"""doc"""', "r'''doc'''", "R'doc'", 'u"doc"'):
            with self.subTest(doc):
                msgids = self.extract_docstrings_from_str(dedent('''\
                %s
                ''' % doc))
                self.assertIn('doc', msgids)

    def test_moduledocstring_bytes(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        b"""doc"""
        '''))
        self.assertFalse([msgid for msgid in msgids if 'doc' in msgid])

    def test_moduledocstring_fstring(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"""doc"""
        '''))
        self.assertFalse([msgid for msgid in msgids if 'doc' in msgid])

    def test_msgid(self):
        msgids = self.extract_docstrings_from_str(
                '''_("""doc""" r'str' u"ing")''')
        self.assertIn('docstring', msgids)

    def test_msgid_bytes(self):
        msgids = self.extract_docstrings_from_str('_(b"""doc""")')
        self.assertFalse([msgid for msgid in msgids if 'doc' in msgid])

    def test_msgid_fstring(self):
        msgids = self.extract_docstrings_from_str('_(f"""doc""")')
        self.assertFalse([msgid for msgid in msgids if 'doc' in msgid])

    def test_funcdocstring_annotated_args(self):
        """ Test docstrings for functions with annotated args """
        msgids = self.extract_docstrings_from_str(dedent('''\
        def foo(bar: str):
            """doc"""
        '''))
        self.assertIn('doc', msgids)

    def test_funcdocstring_annotated_return(self):
        """ Test docstrings for functions with annotated return type """
        msgids = self.extract_docstrings_from_str(dedent('''\
        def foo(bar) -> str:
            """doc"""
        '''))
        self.assertIn('doc', msgids)

    def test_funcdocstring_defvalue_args(self):
        """ Test docstring for functions with default arg values """
        msgids = self.extract_docstrings_from_str(dedent('''\
        def foo(bar=()):
            """doc"""
        '''))
        self.assertIn('doc', msgids)

    def test_funcdocstring_multiple_funcs(self):
        """ Test docstring extraction for multiple functions combining
        annotated args, annotated return types and default arg values
        """
        msgids = self.extract_docstrings_from_str(dedent('''\
        def foo1(bar: tuple=()) -> str:
            """doc1"""

        def foo2(bar: List[1:2]) -> (lambda x: x):
            """doc2"""

        def foo3(bar: 'func'=lambda x: x) -> {1: 2}:
            """doc3"""
        '''))
        self.assertIn('doc1', msgids)
        self.assertIn('doc2', msgids)
        self.assertIn('doc3', msgids)

    def test_classdocstring_early_colon(self):
        """ Test docstring extraction for a class with colons occurring within
        the parentheses.
        """
        msgids = self.extract_docstrings_from_str(dedent('''\
        class D(L[1:2], F({1: 2}), metaclass=M(lambda x: x)):
            """doc"""
        '''))
        self.assertIn('doc', msgids)

    def test_calls_in_fstrings(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"{_('foo bar')}"
        '''))
        self.assertIn('foo bar', msgids)

    def test_calls_in_fstrings_raw(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        rf"{_('foo bar')}"
        '''))
        self.assertIn('foo bar', msgids)

    def test_calls_in_fstrings_nested(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"""{f'{_("foo bar")}'}"""
        '''))
        self.assertIn('foo bar', msgids)

    def test_calls_in_fstrings_attribute(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"{obj._('foo bar')}"
        '''))
        self.assertIn('foo bar', msgids)

    def test_calls_in_fstrings_with_call_on_call(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"{type(str)('foo bar')}"
        '''))
        self.assertNotIn('foo bar', msgids)

    def test_calls_in_fstrings_with_format(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"{_('foo {bar}').format(bar='baz')}"
        '''))
        self.assertIn('foo {bar}', msgids)

    def test_calls_in_fstrings_with_wrong_input_1(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"{_(f'foo {bar}')}"
        '''))
        self.assertFalse([msgid for msgid in msgids if 'foo {bar}' in msgid])

    def test_calls_in_fstrings_with_wrong_input_2(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"{_(1)}"
        '''))
        self.assertNotIn(1, msgids)

    def test_calls_in_fstring_with_multiple_args(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"{_('foo', 'bar')}"
        '''))
        self.assertIn('foo', msgids)
        self.assertNotIn('bar', msgids)

    def test_calls_in_fstring_with_keyword_args(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"{_('foo', bar='baz')}"
        '''))
        self.assertIn('foo', msgids)
        self.assertNotIn('bar', msgids)
        self.assertNotIn('baz', msgids)

    def test_calls_in_fstring_with_partially_wrong_expression(self):
        msgids = self.extract_docstrings_from_str(dedent('''\
        f"{_(f'foo') + _('bar')}"
        '''))
        self.assertNotIn('foo', msgids)
        self.assertIn('bar', msgids)

    def test_function_and_class_names(self):
        """Test that function and class names are not mistakenly extracted."""
        msgids = self.extract_from_str(dedent('''\
        def _(x):
            pass

        def _(x="foo"):
            pass

        async def _(x):
            pass

        class _(object):
            pass
        '''))
        self.assertEqual(msgids, [''])

    def test_pygettext_output(self):
        """Test that the pygettext output exactly matches snapshots."""
        for input_file, output_file, output in extract_from_snapshots():
            with self.subTest(input_file=input_file):
                expected = output_file.read_text(encoding='utf-8')
                self.assert_POT_equal(expected, output)

    def test_files_list(self):
        """Make sure the directories are inspected for source files
           bpo-31920
        """
        text1 = 'Text to translate1'
        text2 = 'Text to translate2'
        text3 = 'Text to ignore'
        with temp_cwd(None), temp_dir(None) as sdir:
            pymod = Path(sdir, 'pypkg', 'pymod.py')
            pymod.parent.mkdir()
            pymod.write_text(f'_({text1!r})', encoding='utf-8')

            pymod2 = Path(sdir, 'pkg.py', 'pymod2.py')
            pymod2.parent.mkdir()
            pymod2.write_text(f'_({text2!r})', encoding='utf-8')

            pymod3 = Path(sdir, 'CVS', 'pymod3.py')
            pymod3.parent.mkdir()
            pymod3.write_text(f'_({text3!r})', encoding='utf-8')

            assert_python_ok('-Xutf8', self.script, sdir)
            data = Path('messages.pot').read_text(encoding='utf-8')
            self.assertIn(f'msgid "{text1}"', data)
            self.assertIn(f'msgid "{text2}"', data)
            self.assertNotIn(text3, data)

    def test_help_text(self):
        """Test that the help text is displayed."""
        res = assert_python_ok(self.script, '--help')
        self.assertEqual(res.out, b'')
        self.assertIn(b'pygettext -- Python equivalent of xgettext(1)', res.err)

    def test_error_messages(self):
        """Test that pygettext outputs error messages to stderr."""
        stderr = self.get_stderr(dedent('''\
        _(1+2)
        ngettext('foo')
        dgettext(*args, 'foo')
        '''))

        # Normalize line endings on Windows
        stderr = stderr.decode('utf-8').replace('\r', '')

        self.assertEqual(
            stderr,
            "*** test.py:1: Expected a string constant for argument 1, got 1 + 2\n"
            "*** test.py:2: Expected at least 2 positional argument(s) in gettext call, got 1\n"
            "*** test.py:3: Variable positional arguments are not allowed in gettext calls\n"
        )

    def test_extract_all_comments(self):
        """
        Test that the --add-comments option without an
        explicit tag extracts all translator comments.
        """
        for arg in ('--add-comments', '-c'):
            with self.subTest(arg=arg):
                data = self.extract_from_str(dedent('''\
                # Translator comment
                _("foo")
                '''), args=(arg,), raw=True)
                self.assertIn('#. Translator comment', data)

    def test_comments_with_multiple_tags(self):
        """
        Test that multiple --add-comments tags can be specified.
        """
        for arg in ('--add-comments={}', '-c{}'):
            with self.subTest(arg=arg):
                args = (arg.format('foo:'), arg.format('bar:'))
                data = self.extract_from_str(dedent('''\
                # foo: comment
                _("foo")

                # bar: comment
                _("bar")

                # baz: comment
                _("baz")
                '''), args=args, raw=True)
                self.assertIn('#. foo: comment', data)
                self.assertIn('#. bar: comment', data)
                self.assertNotIn('#. baz: comment', data)

    def test_comments_not_extracted_without_tags(self):
        """
        Test that translator comments are not extracted without
        specifying --add-comments.
        """
        data = self.extract_from_str(dedent('''\
        # Translator comment
        _("foo")
        '''), raw=True)
        self.assertNotIn('#.', data)

    def test_parse_keyword_spec(self):
        valid = (
            ('foo', ('foo', {'msgid': 0})),
            ('foo:1', ('foo', {'msgid': 0})),
            ('foo:1,2', ('foo', {'msgid': 0, 'msgid_plural': 1})),
            ('foo:1, 2', ('foo', {'msgid': 0, 'msgid_plural': 1})),
            ('foo:1,2c', ('foo', {'msgid': 0, 'msgctxt': 1})),
            ('foo:2c,1', ('foo', {'msgid': 0, 'msgctxt': 1})),
            ('foo:2c ,1', ('foo', {'msgid': 0, 'msgctxt': 1})),
            ('foo:1,2,3c', ('foo', {'msgid': 0, 'msgid_plural': 1, 'msgctxt': 2})),
            ('foo:1, 2, 3c', ('foo', {'msgid': 0, 'msgid_plural': 1, 'msgctxt': 2})),
            ('foo:3c,1,2', ('foo', {'msgid': 0, 'msgid_plural': 1, 'msgctxt': 2})),
        )
        for spec, expected in valid:
            with self.subTest(spec=spec):
                self.assertEqual(parse_spec(spec), expected)
                # test unparse-parse round-trip
                self.assertEqual(parse_spec(unparse_spec(*expected)), expected)

        invalid = (
            ('foo:', "Invalid keyword spec 'foo:': missing argument positions"),
            ('foo:bar', "Invalid keyword spec 'foo:bar': position is not an integer"),
            ('foo:0', "Invalid keyword spec 'foo:0': argument positions must be strictly positive"),
            ('foo:-2', "Invalid keyword spec 'foo:-2': argument positions must be strictly positive"),
            ('foo:1,1', "Invalid keyword spec 'foo:1,1': duplicate positions"),
            ('foo:1,2,1', "Invalid keyword spec 'foo:1,2,1': duplicate positions"),
            ('foo:1c,2,1c', "Invalid keyword spec 'foo:1c,2,1c': duplicate positions"),
            ('foo:1c,2,3c', "Invalid keyword spec 'foo:1c,2,3c': msgctxt can only appear once"),
            ('foo:1,2,3', "Invalid keyword spec 'foo:1,2,3': too many positions"),
            ('foo:1c', "Invalid keyword spec 'foo:1c': msgctxt cannot appear without msgid"),
        )
        for spec, message in invalid:
            with self.subTest(spec=spec):
                with self.assertRaises(ValueError) as cm:
                    parse_spec(spec)
                self.assertEqual(str(cm.exception), message)

    def test_process_keywords(self):
        default_keywords = {name: [spec] for name, spec
                            in DEFAULTKEYWORDS.items()}
        inputs = (
            (['foo'], True),
            (['_:1,2'], True),
            (['foo', 'foo:1,2'], True),
            (['foo'], False),
            (['_:1,2', '_:1c,2,3', 'pgettext'], False),
            # Duplicate entries
            (['foo', 'foo'], True),
            (['_'], False)
        )
        expected = (
            {'foo': [{'msgid': 0}]},
            {'_': [{'msgid': 0, 'msgid_plural': 1}]},
            {'foo': [{'msgid': 0}, {'msgid': 0, 'msgid_plural': 1}]},
            default_keywords | {'foo': [{'msgid': 0}]},
            default_keywords | {'_': [{'msgid': 0, 'msgid_plural': 1},
                                      {'msgctxt': 0, 'msgid': 1, 'msgid_plural': 2},
                                      {'msgid': 0}],
                                'pgettext': [{'msgid': 0},
                                             {'msgctxt': 0, 'msgid': 1}]},
            {'foo': [{'msgid': 0}]},
            default_keywords,
        )
        for (keywords, no_default_keywords), expected in zip(inputs, expected):
            with self.subTest(keywords=keywords,
                              no_default_keywords=no_default_keywords):
                processed = process_keywords(
                    keywords,
                    no_default_keywords=no_default_keywords)
                self.assertEqual(processed, expected)

    def test_multiple_keywords_same_funcname_errors(self):
        # If at least one keyword spec for a given funcname matches,
        # no error should be printed.
        msgids, stderr = self.extract_from_str(dedent('''\
        _("foo", 42)
        _(42, "bar")
        '''), args=('--keyword=_:1', '--keyword=_:2'), with_stderr=True)
        self.assertIn('foo', msgids)
        self.assertIn('bar', msgids)
        self.assertEqual(stderr, b'')

        # If no keyword spec for a given funcname matches,
        # all errors are printed.
        msgids, stderr = self.extract_from_str(dedent('''\
        _(x, 42)
        _(42, y)
        '''), args=('--keyword=_:1', '--keyword=_:2'), with_stderr=True,
              strict=False)
        self.assertEqual(msgids, [''])
        # Normalize line endings on Windows
        stderr = stderr.decode('utf-8').replace('\r', '')
        self.assertEqual(
            stderr,
            '*** test.py:1: No keywords matched gettext call "_":\n'
            '\tkeyword="_": Expected a string constant for argument 1, got x\n'
            '\tkeyword="_:2": Expected a string constant for argument 2, got 42\n'
            '*** test.py:2: No keywords matched gettext call "_":\n'
            '\tkeyword="_": Expected a string constant for argument 1, got 42\n'
            '\tkeyword="_:2": Expected a string constant for argument 2, got y\n')


def extract_from_snapshots():
    snapshots = {
        'messages.py': (),
        'fileloc.py': ('--docstrings',),
        'docstrings.py': ('--docstrings',),
        'comments.py': ('--add-comments=i18n:',),
        'custom_keywords.py': ('--keyword=foo', '--keyword=nfoo:1,2',
                               '--keyword=pfoo:1c,2',
                               '--keyword=npfoo:1c,2,3', '--keyword=_:1,2'),
        'multiple_keywords.py': ('--keyword=foo:1c,2,3', '--keyword=foo:1c,2',
                                 '--keyword=foo:1,2',
                                 # repeat a keyword to make sure it is extracted only once
                                 '--keyword=foo', '--keyword=foo'),
        # == Test character escaping
        # Escape ascii and unicode:
        'escapes.py': ('--escape', '--add-comments='),
        # Escape only ascii and let unicode pass through:
        ('escapes.py', 'ascii-escapes.pot'): ('--add-comments=',),
    }

    for filename, args in snapshots.items():
        if isinstance(filename, tuple):
            filename, output_file = filename
            output_file = DATA_DIR / output_file
            input_file = DATA_DIR / filename
        else:
            input_file = DATA_DIR / filename
            output_file = input_file.with_suffix('.pot')
        contents = input_file.read_bytes()
        with temp_cwd(None):
            Path(input_file.name).write_bytes(contents)
            assert_python_ok('-Xutf8', Test_pygettext.script, *args,
                             input_file.name)
            yield (input_file, output_file,
                   Path('messages.pot').read_text(encoding='utf-8'))


def update_POT_snapshots():
    for _, output_file, output in extract_from_snapshots():
        output = normalize_POT_file(output)
        output_file.write_text(output, encoding='utf-8')


if __name__ == '__main__':
    # To regenerate POT files
    if len(sys.argv) > 1 and sys.argv[1] == '--snapshot-update':
        update_POT_snapshots()
        sys.exit(0)
    unittest.main()

### File: test_makefile.py

In [ ]:
"""
Tests for `Makefile`.
"""

import os
import unittest
from test import support
import sysconfig

MAKEFILE = sysconfig.get_makefile_filename()

if not support.check_impl_detail(cpython=True):
    raise unittest.SkipTest('cpython only')
if not os.path.exists(MAKEFILE) or not os.path.isfile(MAKEFILE):
    raise unittest.SkipTest('Makefile could not be found')


class TestMakefile(unittest.TestCase):
    def list_test_dirs(self):
        result = []
        found_testsubdirs = False
        with open(MAKEFILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.startswith('TESTSUBDIRS='):
                    found_testsubdirs = True
                    result.append(
                        line.removeprefix('TESTSUBDIRS=').replace(
                            '\\', '',
                        ).strip(),
                    )
                    continue
                if found_testsubdirs:
                    if '\t' not in line:
                        break
                    result.append(line.replace('\\', '').strip())
        return result

    @unittest.skipUnless(support.TEST_MODULES_ENABLED, "requires test modules")
    def test_makefile_test_folders(self):
        test_dirs = self.list_test_dirs()
        idle_test = 'idlelib/idle_test'
        self.assertIn(idle_test, test_dirs)

        used = set([idle_test])
        for dirpath, dirs, files in os.walk(support.TEST_HOME_DIR):
            dirname = os.path.basename(dirpath)
            # Skip temporary dirs:
            if dirname == '__pycache__' or dirname.startswith('.'):
                dirs.clear()  # do not process subfolders
                continue
            # Skip empty dirs:
            if not dirs and not files:
                continue
            # Skip dirs with hidden-only files:
            if files and all(
                filename.startswith('.') or filename == '__pycache__'
                for filename in files
            ):
                continue

            relpath = os.path.relpath(dirpath, support.STDLIB_DIR)
            with self.subTest(relpath=relpath):
                self.assertIn(
                    relpath,
                    test_dirs,
                    msg=(
                        f"{relpath!r} is not included in the Makefile's list "
                        "of test directories to install"
                    )
                )
                used.add(relpath)

        # Don't check the wheel dir when Python is built --with-wheel-pkg-dir
        if sysconfig.get_config_var('WHEEL_PKG_DIR'):
            test_dirs.remove('test/wheeldata')
            used.discard('test/wheeldata')

        # Check that there are no extra entries:
        unique_test_dirs = set(test_dirs)
        self.assertSetEqual(unique_test_dirs, used)
        self.assertEqual(len(test_dirs), len(unique_test_dirs))

### File: test_makeunicodedata.py

In [ ]:
import unittest
from test.test_tools import skip_if_missing, imports_under_tool
from test import support
from test.support.hypothesis_helper import hypothesis

st = hypothesis.strategies
given = hypothesis.given
example = hypothesis.example


skip_if_missing("unicode")
with imports_under_tool("unicode"):
    from dawg import Dawg, build_compression_dawg, lookup, inverse_lookup


@st.composite
def char_name_db(draw, min_length=1, max_length=30):
    m = draw(st.integers(min_value=min_length, max_value=max_length))
    names = draw(
        st.sets(st.text("abcd", min_size=1, max_size=10), min_size=m, max_size=m)
    )
    characters = draw(st.sets(st.characters(), min_size=m, max_size=m))
    return list(zip(names, characters))


class TestDawg(unittest.TestCase):
    """Tests for the directed acyclic word graph data structure that is used
    to store the unicode character names in unicodedata. Tests ported from PyPy
    """

    def test_dawg_direct_simple(self):
        dawg = Dawg()
        dawg.insert("a", -4)
        dawg.insert("c", -2)
        dawg.insert("cat", -1)
        dawg.insert("catarr", 0)
        dawg.insert("catnip", 1)
        dawg.insert("zcatnip", 5)
        packed, data, inverse = dawg.finish()

        self.assertEqual(lookup(packed, data, b"a"), -4)
        self.assertEqual(lookup(packed, data, b"c"), -2)
        self.assertEqual(lookup(packed, data, b"cat"), -1)
        self.assertEqual(lookup(packed, data, b"catarr"), 0)
        self.assertEqual(lookup(packed, data, b"catnip"), 1)
        self.assertEqual(lookup(packed, data, b"zcatnip"), 5)
        self.assertRaises(KeyError, lookup, packed, data, b"b")
        self.assertRaises(KeyError, lookup, packed, data, b"catni")
        self.assertRaises(KeyError, lookup, packed, data, b"catnipp")

        self.assertEqual(inverse_lookup(packed, inverse, -4), b"a")
        self.assertEqual(inverse_lookup(packed, inverse, -2), b"c")
        self.assertEqual(inverse_lookup(packed, inverse, -1), b"cat")
        self.assertEqual(inverse_lookup(packed, inverse, 0), b"catarr")
        self.assertEqual(inverse_lookup(packed, inverse, 1), b"catnip")
        self.assertEqual(inverse_lookup(packed, inverse, 5), b"zcatnip")
        self.assertRaises(KeyError, inverse_lookup, packed, inverse, 12)

    def test_forbid_empty_dawg(self):
        dawg = Dawg()
        self.assertRaises(ValueError, dawg.finish)

    @given(char_name_db())
    @example([("abc", "a"), ("abd", "b")])
    @example(
        [
            ("bab", "1"),
            ("a", ":"),
            ("ad", "@"),
            ("b", "<"),
            ("aacc", "?"),
            ("dab", "D"),
            ("aa", "0"),
            ("ab", "F"),
            ("aaa", "7"),
            ("cbd", "="),
            ("abad", ";"),
            ("ac", "B"),
            ("abb", "4"),
            ("bb", "2"),
            ("aab", "9"),
            ("caaaaba", "E"),
            ("ca", ">"),
            ("bbaaa", "5"),
            ("d", "3"),
            ("baac", "8"),
            ("c", "6"),
            ("ba", "A"),
        ]
    )
    @example(
        [
            ("bcdac", "9"),
            ("acc", "g"),
            ("d", "d"),
            ("daabdda", "0"),
            ("aba", ";"),
            ("c", "6"),
            ("aa", "7"),
            ("abbd", "c"),
            ("badbd", "?"),
            ("bbd", "f"),
            ("cc", "@"),
            ("bb", "8"),
            ("daca", ">"),
            ("ba", ":"),
            ("baac", "3"),
            ("dbdddac", "a"),
            ("a", "2"),
            ("cabd", "b"),
            ("b", "="),
            ("abd", "4"),
            ("adcbd", "5"),
            ("abc", "e"),
            ("ab", "1"),
        ]
    )
    def test_dawg(self, data):
        # suppress debug prints
        with support.captured_stdout() as output:
            # it's enough to build it, building will also check the result
            build_compression_dawg(data)

### File: test_msgfmt.py

In [ ]:
"""Tests for the Tools/i18n/msgfmt.py tool."""

import json
import struct
import sys
import unittest
from gettext import GNUTranslations
from pathlib import Path

from test.support.os_helper import temp_cwd
from test.support.script_helper import assert_python_failure, assert_python_ok
from test.test_tools import imports_under_tool, skip_if_missing, toolsdir


skip_if_missing('i18n')

data_dir = (Path(__file__).parent / 'msgfmt_data').resolve()
script_dir = Path(toolsdir) / 'i18n'
msgfmt_py = script_dir / 'msgfmt.py'

with imports_under_tool("i18n"):
    import msgfmt


def compile_messages(po_file, mo_file):
    assert_python_ok(msgfmt_py, '-o', mo_file, po_file)


class CompilationTest(unittest.TestCase):

    def test_compilation(self):
        self.maxDiff = None
        with temp_cwd():
            for po_file in data_dir.glob('*.po'):
                with self.subTest(po_file=po_file):
                    mo_file = po_file.with_suffix('.mo')
                    with open(mo_file, 'rb') as f:
                        expected = GNUTranslations(f)

                    tmp_mo_file = mo_file.name
                    compile_messages(po_file, tmp_mo_file)
                    with open(tmp_mo_file, 'rb') as f:
                        actual = GNUTranslations(f)

                    self.assertDictEqual(actual._catalog, expected._catalog)

    def test_binary_header(self):
        with temp_cwd():
            tmp_mo_file = 'messages.mo'
            compile_messages(data_dir / "general.po", tmp_mo_file)
            with open(tmp_mo_file, 'rb') as f:
                mo_data = f.read()

        (
            magic,
            version,
            num_strings,
            orig_table_offset,
            trans_table_offset,
            hash_table_size,
            hash_table_offset,
        ) = struct.unpack("=7I", mo_data[:28])

        self.assertEqual(magic, 0x950412de)
        self.assertEqual(version, 0)
        self.assertEqual(num_strings, 9)
        self.assertEqual(orig_table_offset, 28)
        self.assertEqual(trans_table_offset, 100)
        self.assertEqual(hash_table_size, 0)
        self.assertEqual(hash_table_offset, 0)

    def test_translations(self):
        with open(data_dir / 'general.mo', 'rb') as f:
            t = GNUTranslations(f)

        self.assertEqual(t.gettext('foo'), 'foo')
        self.assertEqual(t.gettext('bar'), 'baz')
        self.assertEqual(t.pgettext('abc', 'foo'), 'bar')
        self.assertEqual(t.pgettext('xyz', 'foo'), 'bar')
        self.assertEqual(t.gettext('Multilinestring'), 'Multilinetranslation')
        self.assertEqual(t.gettext('"escapes"'), '"translated"')
        self.assertEqual(t.gettext('\n newlines \n'), '\n translated \n')
        self.assertEqual(t.ngettext('One email sent.', '%d emails sent.', 1),
                         'One email sent.')
        self.assertEqual(t.ngettext('One email sent.', '%d emails sent.', 2),
                         '%d emails sent.')
        self.assertEqual(t.npgettext('abc', 'One email sent.',
                                     '%d emails sent.', 1),
                         'One email sent.')
        self.assertEqual(t.npgettext('abc', 'One email sent.',
                                     '%d emails sent.', 2),
                         '%d emails sent.')

    def test_po_with_bom(self):
        with temp_cwd():
            Path('bom.po').write_bytes(b'\xef\xbb\xbfmsgid "Python"\nmsgstr "Pioton"\n')

            res = assert_python_failure(msgfmt_py, 'bom.po')
            err = res.err.decode('utf-8')
            self.assertIn('The file bom.po starts with a UTF-8 BOM', err)

    def test_invalid_msgid_plural(self):
        with temp_cwd():
            Path('invalid.po').write_text('''\
msgid_plural "plural"
msgstr[0] "singular"
''')

            res = assert_python_failure(msgfmt_py, 'invalid.po')
            err = res.err.decode('utf-8')
            self.assertIn('msgid_plural not preceded by msgid', err)

    def test_plural_without_msgid_plural(self):
        with temp_cwd():
            Path('invalid.po').write_text('''\
msgid "foo"
msgstr[0] "bar"
''')

            res = assert_python_failure(msgfmt_py, 'invalid.po')
            err = res.err.decode('utf-8')
            self.assertIn('plural without msgid_plural', err)

    def test_indexed_msgstr_without_msgid_plural(self):
        with temp_cwd():
            Path('invalid.po').write_text('''\
msgid "foo"
msgid_plural "foos"
msgstr "bar"
''')

            res = assert_python_failure(msgfmt_py, 'invalid.po')
            err = res.err.decode('utf-8')
            self.assertIn('indexed msgstr required for plural', err)

    def test_generic_syntax_error(self):
        with temp_cwd():
            Path('invalid.po').write_text('''\
"foo"
''')

            res = assert_python_failure(msgfmt_py, 'invalid.po')
            err = res.err.decode('utf-8')
            self.assertIn('Syntax error', err)


class POParserTest(unittest.TestCase):
    @classmethod
    def tearDownClass(cls):
        # msgfmt uses a global variable to store messages,
        # clear it after the tests.
        msgfmt.MESSAGES.clear()

    def test_strings(self):
        # Test that the PO parser correctly handles and unescape
        # strings in the PO file.
        # The PO file format allows for a variety of escape sequences,
        # octal and hex escapes.
        valid_strings = (
            # empty strings
            ('""', ''),
            ('"" "" ""', ''),
            # allowed escape sequences
            (r'"\\"', '\\'),
            (r'"\""', '"'),
            (r'"\t"', '\t'),
            (r'"\n"', '\n'),
            (r'"\r"', '\r'),
            (r'"\f"', '\f'),
            (r'"\a"', '\a'),
            (r'"\b"', '\b'),
            (r'"\v"', '\v'),
            # non-empty strings
            ('"foo"', 'foo'),
            ('"foo" "bar"', 'foobar'),
            ('"foo""bar"', 'foobar'),
            ('"" "foo" ""', 'foo'),
            # newlines and tabs
            (r'"foo\nbar"', 'foo\nbar'),
            (r'"foo\n" "bar"', 'foo\nbar'),
            (r'"foo\tbar"', 'foo\tbar'),
            (r'"foo\t" "bar"', 'foo\tbar'),
            # escaped quotes
            (r'"foo\"bar"', 'foo"bar'),
            (r'"foo\"" "bar"', 'foo"bar'),
            (r'"foo\\" "bar"', 'foo\\bar'),
            # octal escapes
            (r'"\120\171\164\150\157\156"', 'Python'),
            (r'"\120\171\164" "\150\157\156"', 'Python'),
            (r'"\"\120\171\164" "\150\157\156\""', '"Python"'),
            # hex escapes
            (r'"\x50\x79\x74\x68\x6f\x6e"', 'Python'),
            (r'"\x50\x79\x74" "\x68\x6f\x6e"', 'Python'),
            (r'"\"\x50\x79\x74" "\x68\x6f\x6e\""', '"Python"'),
        )

        with temp_cwd():
            for po_string, expected in valid_strings:
                with self.subTest(po_string=po_string):
                    # Construct a PO file with a single entry,
                    # compile it, read it into a catalog and
                    # check the result.
                    po = f'msgid {po_string}\nmsgstr "translation"'
                    Path('messages.po').write_text(po)
                    # Reset the global MESSAGES dictionary
                    msgfmt.MESSAGES.clear()
                    msgfmt.make('messages.po', 'messages.mo')

                    with open('messages.mo', 'rb') as f:
                        actual = GNUTranslations(f)

                    self.assertDictEqual(actual._catalog, {expected: 'translation'})

        invalid_strings = (
            # "''",  # invalid but currently accepted
            '"',
            '"""',
            '"" "',
            'foo',
            '"" "foo',
            '"foo" foo',
            '42',
            '"" 42 ""',
            # disallowed escape sequences
            # r'"\'"',  # invalid but currently accepted
            # r'"\e"',  # invalid but currently accepted
            # r'"\8"',  # invalid but currently accepted
            # r'"\9"',  # invalid but currently accepted
            r'"\x"',
            r'"\u1234"',
            r'"\N{ROMAN NUMERAL NINE}"'
        )
        with temp_cwd():
            for invalid_string in invalid_strings:
                with self.subTest(string=invalid_string):
                    po = f'msgid {invalid_string}\nmsgstr "translation"'
                    Path('messages.po').write_text(po)
                    # Reset the global MESSAGES dictionary
                    msgfmt.MESSAGES.clear()
                    with self.assertRaises(Exception):
                        msgfmt.make('messages.po', 'messages.mo')


class CLITest(unittest.TestCase):

    def test_help(self):
        for option in ('--help', '-h'):
            res = assert_python_ok(msgfmt_py, option)
            err = res.err.decode('utf-8')
            self.assertIn('Generate binary message catalog from textual translation description.', err)

    def test_version(self):
        for option in ('--version', '-V'):
            res = assert_python_ok(msgfmt_py, option)
            out = res.out.decode('utf-8').strip()
            self.assertEqual('msgfmt.py 1.2', out)

    def test_invalid_option(self):
        res = assert_python_failure(msgfmt_py, '--invalid-option')
        err = res.err.decode('utf-8')
        self.assertIn('Generate binary message catalog from textual translation description.', err)
        self.assertIn('option --invalid-option not recognized', err)

    def test_no_input_file(self):
        res = assert_python_ok(msgfmt_py)
        err = res.err.decode('utf-8').replace('\r\n', '\n')
        self.assertIn('No input file given\n'
                      "Try `msgfmt --help' for more information.", err)

    def test_nonexistent_file(self):
        assert_python_failure(msgfmt_py, 'nonexistent.po')


def update_catalog_snapshots():
    for po_file in data_dir.glob('*.po'):
        mo_file = po_file.with_suffix('.mo')
        compile_messages(po_file, mo_file)
        # Create a human-readable JSON file which is
        # easier to review than the binary .mo file.
        with open(mo_file, 'rb') as f:
            translations = GNUTranslations(f)
        catalog_file = po_file.with_suffix('.json')
        with open(catalog_file, 'w') as f:
            data = translations._catalog.items()
            data = sorted(data, key=lambda x: (isinstance(x[0], tuple), x[0]))
            json.dump(data, f, indent=4)
            f.write('\n')


if __name__ == '__main__':
    if len(sys.argv) > 1 and sys.argv[1] == '--snapshot-update':
        update_catalog_snapshots()
        sys.exit(0)
    unittest.main()

### File: test_reindent.py

In [ ]:
"""Tests for scripts in the Tools directory.

This file contains regression tests for some of the scripts found in the
Tools directory of a Python checkout or tarball, such as reindent.py.
"""

import os
import unittest
from test.support.script_helper import assert_python_ok
from test.support import findfile

from test.test_tools import toolsdir, skip_if_missing

skip_if_missing()

class ReindentTests(unittest.TestCase):
    script = os.path.join(toolsdir, 'patchcheck', 'reindent.py')

    def test_noargs(self):
        assert_python_ok(self.script)

    def test_help(self):
        rc, out, err = assert_python_ok(self.script, '-h')
        self.assertEqual(out, b'')
        self.assertGreater(err, b'')

    def test_reindent_file_with_bad_encoding(self):
        bad_coding_path = findfile('bad_coding.py', subdir='tokenizedata')
        rc, out, err = assert_python_ok(self.script, '-r', bad_coding_path)
        self.assertEqual(out, b'')
        self.assertNotEqual(err, b'')


if __name__ == '__main__':
    unittest.main()

### File: test_sundry.py

In [ ]:
"""Tests for scripts in the Tools/scripts directory.

This file contains extremely basic regression tests for the scripts found in
the Tools directory of a Python checkout or tarball which don't have separate
tests of their own.
"""

import os
import unittest
from test.support import import_helper

from test.test_tools import scriptsdir, import_tool, skip_if_missing

skip_if_missing()

class TestSundryScripts(unittest.TestCase):
    # import logging registers "atfork" functions which keep indirectly the
    # logging module dictionary alive. Mock the function to be able to unload
    # cleanly the logging module.
    @import_helper.mock_register_at_fork
    def test_sundry(self, mock_os):
        for fn in os.listdir(scriptsdir):
            if not fn.endswith('.py'):
                continue
            name = fn[:-3]
            import_tool(name)


if __name__ == '__main__':
    unittest.main()